# Mount Drive

In [17]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## I'm Defining paths

In [18]:
import os, pathlib

DRIVE_DATA_DIR = "/content/drive/MyDrive/DATA 255 Lab-1"
WORK_DIR = "/content/data255_lab1"
os.makedirs(WORK_DIR, exist_ok=True)

train_zip = f"{DRIVE_DATA_DIR}/imagenet_train20.zip"
val_zip   = f"{DRIVE_DATA_DIR}/imagenet_val20.zip"

print("train_zip exists:", os.path.exists(train_zip))
print("val_zip exists:", os.path.exists(val_zip))
print("WORK_DIR:", WORK_DIR)

train_zip exists: True
val_zip exists: True
WORK_DIR: /content/data255_lab1


In [3]:
!pip -q install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 20.7 MB/s eta 0:00:00


## Unzip to /content (fast local disk)

In [19]:
!rm -rf /content/imagenet_train20*
!rm -rf /content/imagenet_val20*
!unzip -q "$train_zip" -d /content/
!unzip -q "$val_zip" -d /content/
!ls /content | grep -i imagenet

imagenet_train20a
imagenet_val20


In [5]:
import random, time
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [20]:
import os

candidates = [x for x in os.listdir("/content") if "imagenet" in x.lower()]
print("Candidates:", candidates)

train_root = None
val_root = None

for x in candidates:
    if "train" in x.lower():
        train_root = "/content/" + x
    if "val" in x.lower():
        val_root = "/content/" + x

print("train_root:", train_root)
print("val_root:", val_root)

Candidates: ['imagenet_val20', 'imagenet_train20a']
train_root: /content/imagenet_train20a
val_root: /content/imagenet_val20


In [21]:
import os

print("Train top-level items:", len(os.listdir(train_root)))   # should be 20 folders
print("Val top-level items:", len(os.listdir(val_root)))       # usually 1000 images

first_class = sorted(os.listdir(train_root))[0]
print("Example class folder:", first_class)
print("Images in that class:", len(os.listdir(os.path.join(train_root, first_class))))

Train top-level items: 20
Val top-level items: 1000
Example class folder: n01737021
Images in that class: 300


## GPU Check

In [22]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [25]:
import os

# Set correct path to your validation txt file in Drive
val_txt = "/content/drive/MyDrive/DATA 255 Lab-1/imagenet_val20.txt"

print("val_txt exists:", os.path.exists(val_txt))

val_txt exists: True


## Dataset + Transforms (240×240) + DataLoaders

In [26]:
print("val_root:", val_root)
print("val txt exists?", os.path.exists(val_txt))
print("val images count:", len([f for f in os.listdir(val_root) if f.lower().endswith(("jpg","jpeg","png","jpeg","JPEG","JPG"))]))
print("val_root sample:", os.listdir(val_root)[:10])

val_root: /content/imagenet_val20
val txt exists? True
val images count: 1000
val_root sample: ['n01737021_2519.JPEG', 'n03721384_32601.JPEG', 'n02011460_2719.JPEG', 'n02415577_21084.JPEG', 'n03016953_6410.JPEG', 'n03016953_11692.JPEG', 'n03935335_14021.JPEG', 'n01737021_4667.JPEG', 'n02808440_4617.JPEG', 'n03721384_6217.JPEG']


In [27]:
from torchvision import datasets

tmp_train = datasets.ImageFolder(train_root)   # only to get class order
class_to_idx = tmp_train.class_to_idx          # wnid -> index 0..19
idx_to_class = {v:k for k,v in class_to_idx.items()}

print("num classes:", len(class_to_idx))
print("example:", list(class_to_idx.items())[:5])

num classes: 20
example: [('n01737021', 0), ('n02006656', 1), ('n02011460', 2), ('n02013706', 3), ('n02033041', 4)]


In [28]:
from torch.utils.data import Dataset
from PIL import Image

class ValTxtDataset(Dataset):
    def __init__(self, img_dir, txt_path, transform, class_to_idx):
        self.img_dir = img_dir
        self.transform = transform
        self.samples = []

        with open(txt_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                fname, label = line.split()
                # label in txt is 0..19 (your example shows that)
                y = int(label)
                path = os.path.join(img_dir, fname)
                self.samples.append((path, y))

        # sanity check: first few paths exist
        missing = [p for p,_ in self.samples[:20] if not os.path.exists(p)]
        if missing:
            print("WARNING missing files example:", missing[:3])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, y = self.samples[i]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y

In [29]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os

IMG_SIZE = 240
BATCH_SIZE = 64


train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225]),
])

train_ds = datasets.ImageFolder(train_root, transform=train_tfms)

# Get class mapping from train
class_to_idx = train_ds.class_to_idx


val_txt = "/content/drive/MyDrive/DATA 255 Lab-1/imagenet_val20.txt"

class ValTxtDataset(Dataset):
    def __init__(self, img_dir, txt_path, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.samples = []

        with open(txt_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                fname, label = line.split()
                path = os.path.join(img_dir, fname)
                self.samples.append((path, int(label)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

val_ds = ValTxtDataset(val_root, val_txt, transform=val_tfms)


print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print("Num classes:", len(train_ds.classes))


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)

Train size: 6000
Val size: 1000
Num classes: 20


In [30]:
x, y = next(iter(train_loader))
print(x.shape, y.shape, y[:10])

torch.Size([64, 3, 240, 240]) torch.Size([64]) tensor([ 0,  6, 12,  9, 11, 10,  9,  7, 15,  5])


### Define a custom CNN (not ResNet)

In [31]:
import torch.nn as nn
import torch.nn.functional as F

class MyNet20(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()
        def conv_bn(in_ch, out_ch, k=3, s=1, p=1):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )

        self.stem = nn.Sequential(
            conv_bn(3, 32, 3, 2, 1),   # 240 -> 120
            conv_bn(32, 32),
            nn.MaxPool2d(2)            # 120 -> 60
        )

        self.block1 = nn.Sequential(
            conv_bn(32, 64, 3, 2, 1),  # 60 -> 30
            conv_bn(64, 64),
        )
        self.block2 = nn.Sequential(
            conv_bn(64, 128, 3, 2, 1), # 30 -> 15
            conv_bn(128, 128),
        )
        self.block3 = nn.Sequential(
            conv_bn(128, 192, 3, 2, 1), # 15 -> 8
            conv_bn(192, 192),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(192, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.head(x)

model = MyNet20(num_classes=len(train_ds.classes)).to(device)

with torch.no_grad():
    dummy = torch.randn(2,3,IMG_SIZE,IMG_SIZE).to(device)
    out = model(dummy)
print("Output shape:", out.shape)

Output shape: torch.Size([2, 20])


### Train/Eval Functions

In [32]:
import torch, time

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    correct = total = 0
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return running_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    correct = total = 0
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return running_loss/total, correct/total

### Train (saves best_model.pth to Drive)

In [33]:
import torch.optim as optim
import os

OUT_DIR = "/content/drive/MyDrive/DATA 255 Lab-1/outputs_lab1"
os.makedirs(OUT_DIR, exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_acc = 0.0
best_path = f"{OUT_DIR}/best_model.pth"

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(model.state_dict(), best_path)

    print(f"Epoch {epoch:02d} | train_acc {tr_acc*100:.2f}% | val_acc {va_acc*100:.2f}% | best {best_acc*100:.2f}% | {time.time()-t0:.1f}s", flush=True)

print("Saved best_model.pth to:", best_path)

KeyboardInterrupt: 